<a href="https://colab.research.google.com/github/asukaa2/B-LoRA/blob/main/B_LoRA_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B-LoRA — Inference Demo

Uses the **modern diffusers (PEFT) LoRA backend** — compatible with Python 3.10–3.13.
For training your own B-LoRAs, see `B_LoRA_training.ipynb`.

In [ ]:
!pip install -q -U diffusers peft accelerate transformers safetensors

In [ ]:
!git clone https://github.com/asukaa2/B-LoRA.git
%cd B-LoRA

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from blora_utils import BLOCKS, filter_lora, scale_lora, load_lora_state_dict
import matplotlib.pyplot as plt

def plot_images(images, titles=None, figsize=(15, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for i, img in enumerate(images):
        axes[i].imshow(img)
        if titles:
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    plt.show()

In [ ]:
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipeline = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae,
        torch_dtype=torch.float16,
    ).to("cuda")

## Inference
In the following code, we will show few examples of how to utilize B-LoRA.
You can use our B-LoRAs from the HF 🤗 library `lora-library` in the following template: `lora-library/B-LoRA-<Name>`.
Or you can train your own B-LoRAs (see `B_LoRA_training.ipynb`)

| Image | Name  | Instance Prompt |
|-----------|-----------|-----------|
| <img src="https://b-lora.github.io/B-LoRA/static/figures/content_images/bull.jpg" alt="bull" width="200" height="200"> | <font size="5">bull</font> | <font size="5">A [v3]</font> |
| <img src="https://b-lora.github.io/B-LoRA/static/figures/content_images/teddybear.jpg" alt="teddybear" width="200" height="200"> | <font size="5">teddybear</font> | <font size="5">A [v18]</font> |
| <img src="https://b-lora.github.io/B-LoRA/static/figures/content_images/wolf_plushie.jpg" alt="wolf_plushie" width="200" height="200"> | <font size="5">wolf_plushie</font> | <font size="5">A [v38]</font> |
| <img src="https://b-lora.github.io/B-LoRA/static/figures/content_images/dog2.jpg" alt="dog2" width="200" height="200"> | <font size="5">dog2</font> | <font size="5">A [v] dog</font> |

Note that the last example of `dog2` is different, as it was trained on multiple images for recontextualization, as we will demonstrate next.

![apps method](https://b-lora.github.io/B-LoRA/static/figures/apps_method1.png)

In [ ]:
def load_b_lora_to_unet(pipe, content_lora_model_id: str = '', style_lora_model_id: str = '', content_alpha: float = 1.,
                            style_alpha: float = 1.) -> None:
        try:
            # Get Content B-LoRA SD (`load_lora_state_dict` accepts a local path/dir or a HF repo id)
            if content_lora_model_id:
                content_B_LoRA_sd = load_lora_state_dict(content_lora_model_id)
                content_B_LoRA = filter_lora(content_B_LoRA_sd, BLOCKS['content'])
                content_B_LoRA = scale_lora(content_B_LoRA, content_alpha)
            else:
                content_B_LoRA = {}

            # Get Style B-LoRA SD
            if style_lora_model_id:
                style_B_LoRA_sd = load_lora_state_dict(style_lora_model_id)
                style_B_LoRA = filter_lora(style_B_LoRA_sd, BLOCKS['style'])
                style_B_LoRA = scale_lora(style_B_LoRA, style_alpha)
            else:
                style_B_LoRA = {}

            # Merge B-LoRAs SD
            res_lora = {**content_B_LoRA, **style_B_LoRA}

            # Remove a previously loaded 'blora' adapter (if any), then load the new one.
            if 'blora' in getattr(pipe.unet, 'peft_config', {}):
                pipe.delete_adapters('blora')
            pipe.load_lora_weights(res_lora, adapter_name='blora')
        except Exception as e:
            raise type(e)(f'failed to load_b_lora_to_unet, due to: {e}')

## Image Stylization Based On Image Style Reference

In [ ]:
content_B_LoRA_path = 'lora-library/B-LoRA-teddybear'
style_B_LoRA_path = 'lora-library/B-LoRA-pen_sketch'
content_alpha,style_alpha = 1,1.1

In [ ]:
load_b_lora_to_unet(pipeline, content_B_LoRA_path, style_B_LoRA_path, content_alpha, style_alpha)

In [ ]:
prompt = 'A [v18] in [v30] style'
image = pipeline(prompt,generator=torch.Generator(device="cuda").manual_seed(48), num_images_per_prompt=1).images[0].resize((512,512))
image

### Unload B-LoRA

In [ ]:
pipeline.delete_adapters('blora')
# or fully reset all LoRA layers:
# pipeline.unload_lora_weights()

## Image Stylization Based On Text Reference

In [ ]:
content_B_LoRA_path = 'lora-library/B-LoRA-bull'
style_B_LoRA_path = ''
content_alpha,style_alpha = 1,1

In [ ]:
load_b_lora_to_unet(pipeline, content_B_LoRA_path, style_B_LoRA_path, content_alpha, style_alpha)

In [ ]:
prompt = 'A [v3] made of gold'
image = pipeline(prompt,generator=torch.Generator(device="cuda").manual_seed(48), num_images_per_prompt=1).images[0].resize((512,512))
image

### Unload B-LoRA

In [ ]:
pipeline.delete_adapters('blora')

## Consistent Style Generation

In [ ]:
content_B_LoRA_path = ''
style_B_LoRA_path = 'lora-library/B-LoRA-wolf_plushie'
content_alpha,style_alpha = 1,1.1

In [ ]:
load_b_lora_to_unet(pipeline, content_B_LoRA_path, style_B_LoRA_path, content_alpha, style_alpha)

In [ ]:
prompt = 'A car in [v38] style'
image = pipeline(prompt, generator=torch.Generator(device="cuda").manual_seed(48), num_images_per_prompt=1).images[0].resize((512,512))
image

### Unload B-LoRA

In [ ]:
pipeline.delete_adapters('blora')

## B-LoRA for Personalization

In [ ]:
content_B_LoRA_path = 'lora-library/B-LoRA-multi-dog2'
style_B_LoRA_path = 'lora-library/B-LoRA-wolf_plushie'
content_alpha,style_alpha = 1,1.1

In [ ]:
load_b_lora_to_unet(pipeline, content_B_LoRA_path, style_B_LoRA_path, content_alpha, style_alpha)

In [ ]:
prompt = 'A [v] dog playing with a ball in [v38] style'
image = pipeline(prompt,generator=torch.Generator(device="cuda").manual_seed(12), num_images_per_prompt=1).images[0].resize((512,512))
image

### Unload B-LoRA

In [ ]:
pipeline.delete_adapters('blora')
pipeline.unload_lora_weights()